# 🎯 Fitts' Law Experiment Analysis

Analyze head-controlled pointing data from multiple participants.

## 🚀 How to Use:
1. Upload your `fitts-student-data` folder (zip it first)
2. Run all cells (Runtime → Run all)
3. View results and download graphs

In [ ]:
# 1️⃣ Setup - Install packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Packages loaded!")

In [ ]:
# 2️⃣ Upload Data
from google.colab import files
import zipfile

print("📁 Upload your fitts-student-data.zip file:")
uploaded = files.upload()

# Extract if zip
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted {filename}")

In [ ]:
# 3️⃣ Load Data
import glob

raw_files = glob.glob('**/fitts-experiment-raw-data*.csv', recursive=True)
results_files = glob.glob('**/fitts-experiment-results*.csv', recursive=True)

print(f"Found {len(raw_files)} raw files, {len(results_files)} results files")

# Load raw data
raw_list = []
for i, file in enumerate(raw_files, 1):
    df = pd.read_csv(file)
    participant = file.split('/')[0] if '/' in file else f'P{i}'
    df['ParticipantID'] = participant
    raw_list.append(df)
    print(f"  ✅ {participant}: {len(df)} trials")

raw_data = pd.concat(raw_list, ignore_index=True)
print(f"\n✅ Total: {len(raw_data)} trials from {raw_data['ParticipantID'].nunique()} participants")

# Load results if available
if results_files:
    results_list = []
    for file in results_files:
        df = pd.read_csv(file)
        df['ParticipantID'] = file.split('/')[0] if '/' in file else 'Unknown'
        results_list.append(df)
    results_data = pd.concat(results_list, ignore_index=True)
else:
    results_data = None

In [ ]:
# 4️⃣ Calculate Metrics (if needed)
if results_data is None:
    print("📊 Calculating Fitts metrics...")
    results = []
    
    for (pid, ftype, tsize, amp), group in raw_data.groupby(['ParticipantID', 'FilterType', 'TargetSize', 'Amplitude']):
        Ae = group['ActualAmplitude'].mean()
        # ISO 9241-9 directional projection: project endpoint deviation onto movement direction
        if 'Direction' in group.columns and 'TargetX' in group.columns:
            theta_rad = np.radians(group['Direction'].astype(float))
            dx = group['SelectionX'] - group['TargetX']
            dy = group['SelectionY'] - group['TargetY']
            projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
            We = 4.133 * projections.std()
        else:
            dx = group['SelectionX'] - group['TargetX']
            dy = group['SelectionY'] - group['TargetY']
            distances = np.sqrt(dx**2 + dy**2)
            We = 4.133 * distances.std()
        IDe = np.log2(Ae / We + 1) if We > 0 else 0
        MeanMT = group['MovementTime'].mean()
        TP = IDe / MeanMT if MeanMT > 0 else 0
        
        results.append({
            'ParticipantID': pid,
            'FilterType': ftype,
            'TargetSize': tsize,
            'Amplitude': amp,
            'N': len(group),
            'MeanMT': MeanMT,
            'Ae': Ae,
            'We': We,
            'IDe': IDe,
            'TP': TP
        })
    
    results_data = pd.DataFrame(results)
    print(f"✅ Calculated {len(results)} conditions")

print("\n📊 Results preview:")
display(results_data.head())

In [ ]:
# 5️⃣ Summary Statistics
print("="*80)
print("📊 SUMMARY STATISTICS")
print("="*80)
print(f"\n👥 Participants: {results_data['ParticipantID'].nunique()}")
print(f"📊 Total Trials: {len(raw_data)}")
print(f"\n🎯 Overall:")
print(f"   Throughput: {results_data['TP'].mean():.3f} ± {results_data['TP'].std():.3f} bits/s")
print(f"   Movement Time: {results_data['MeanMT'].mean():.3f} ± {results_data['MeanMT'].std():.3f} s")

print(f"\n🔧 By Filter:")
filter_stats = results_data.groupby('FilterType').agg({
    'TP': ['mean', 'std'],
    'MeanMT': ['mean', 'std']
}).round(3)
display(filter_stats)

In [ ]:
# 6️⃣ Statistical Tests
print("="*80)
print("📊 STATISTICAL TESTS")
print("="*80)

filters = results_data['FilterType'].unique()
if len(filters) == 2:
    f1, f2 = filters
    
    # Get throughput by participant
    tp1 = results_data[results_data['FilterType'] == f1].groupby('ParticipantID')['TP'].mean()
    tp2 = results_data[results_data['FilterType'] == f2].groupby('ParticipantID')['TP'].mean()
    
    print(f"\n📈 Throughput:")
    print(f"   {f1}: {tp1.mean():.3f} ± {tp1.std():.3f}")
    print(f"   {f2}: {tp2.mean():.3f} ± {tp2.std():.3f}")
    
    # Paired t-test
    common = set(tp1.index) & set(tp2.index)
    if len(common) > 1:
        tp1_vals = [tp1[p] for p in common]
        tp2_vals = [tp2[p] for p in common]
        t_stat, p_val = stats.ttest_rel(tp1_vals, tp2_vals)
        
        print(f"\n   t-test: t({len(common)-1})={t_stat:.3f}, p={p_val:.4f}")
        
        if p_val < 0.05:
            better = f1 if np.mean(tp1_vals) > np.mean(tp2_vals) else f2
            improvement = abs((np.mean(tp1_vals) - np.mean(tp2_vals)) / np.mean(tp1_vals) * 100)
            print(f"   ✅ Significant! {better} is {improvement:.1f}% better")
        else:
            print(f"   ❌ Not significant (p >= 0.05)")
        
        # Cohen's d
        diff = np.array(tp1_vals) - np.array(tp2_vals)
        cohens_d = np.mean(diff) / np.std(diff, ddof=1)
        print(f"   Effect size (d): {cohens_d:.3f}")

In [ ]:
# 7️⃣ Graph 1: Throughput by Filter
plt.figure(figsize=(10, 6))
sns.boxplot(data=results_data, x='FilterType', y='TP', palette='Set2')
sns.swarmplot(data=results_data, x='FilterType', y='TP', color='black', alpha=0.3)
plt.title('Throughput by Filter Type', fontsize=16, fontweight='bold')
plt.ylabel('Throughput (bits/s)')
plt.xlabel('Filter Type')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('throughput_by_filter.png', dpi=300)
plt.show()
print("💾 Saved: throughput_by_filter.png")

In [ ]:
# 8️⃣ Graph 2: Movement Time by Filter
plt.figure(figsize=(10, 6))
sns.boxplot(data=results_data, x='FilterType', y='MeanMT', palette='Set2')
sns.swarmplot(data=results_data, x='FilterType', y='MeanMT', color='black', alpha=0.3)
plt.title('Movement Time by Filter Type', fontsize=16, fontweight='bold')
plt.ylabel('Movement Time (s)')
plt.xlabel('Filter Type')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('movement_time_by_filter.png', dpi=300)
plt.show()
print("💾 Saved: movement_time_by_filter.png")

In [ ]:
# 9️⃣ Graph 3: Participant Performance
participant_tp = results_data.groupby(['ParticipantID', 'FilterType'])['TP'].mean().reset_index()

plt.figure(figsize=(14, 6))
sns.barplot(data=participant_tp, x='ParticipantID', y='TP', hue='FilterType', palette='Set2')
plt.title('Individual Participant Throughput', fontsize=16, fontweight='bold')
plt.ylabel('Throughput (bits/s)')
plt.xlabel('Participant')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Filter')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('participant_throughput.png', dpi=300)
plt.show()
print("💾 Saved: participant_throughput.png")

In [ ]:
# 🔟 Graph 4: Fitts' Law Regression
plt.figure(figsize=(10, 6))

for ftype in results_data['FilterType'].unique():
    data = results_data[results_data['FilterType'] == ftype]
    plt.scatter(data['IDe'], data['MeanMT'], label=ftype, alpha=0.6, s=50)
    
    # Regression line
    z = np.polyfit(data['IDe'], data['MeanMT'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(data['IDe'].min(), data['IDe'].max(), 100)
    plt.plot(x_line, p(x_line), '--', linewidth=2)

plt.xlabel('Index of Difficulty (bits)')
plt.ylabel('Movement Time (s)')
plt.title("Fitts' Law: MT = a + b × ID", fontsize=16, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fitts_law_regression.png', dpi=300)
plt.show()
print("💾 Saved: fitts_law_regression.png")

In [ ]:
# 1️⃣1️⃣ Graph 5: Performance by Target Size
plt.figure(figsize=(12, 6))
sns.barplot(data=results_data, x='TargetSize', y='TP', hue='FilterType', palette='Set2')
plt.title('Throughput by Target Size', fontsize=16, fontweight='bold')
plt.ylabel('Throughput (bits/s)')
plt.xlabel('Target Size (pixels)')
plt.legend(title='Filter')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('throughput_by_size.png', dpi=300)
plt.show()
print("💾 Saved: throughput_by_size.png")

In [ ]:
# 1️⃣2️⃣ Export CSV Files
# Summary by filter
summary_filter = results_data.groupby('FilterType').agg({
    'TP': ['mean', 'std', 'min', 'max'],
    'MeanMT': ['mean', 'std', 'min', 'max']
}).round(4)
summary_filter.to_csv('summary_by_filter.csv')
print("💾 Saved: summary_by_filter.csv")

# Summary by participant
summary_participant = results_data.groupby(['ParticipantID', 'FilterType']).agg({
    'TP': 'mean',
    'MeanMT': 'mean'
}).round(4)
summary_participant.to_csv('summary_by_participant.csv')
print("💾 Saved: summary_by_participant.csv")

# All results
results_data.to_csv('all_results.csv', index=False)
print("💾 Saved: all_results.csv")

print("\n✅ Done! Download files from the Files panel (left sidebar)")